# 🔐 CyberNews Scraper — Google Colab

Pipeline para recopilar, puntuar y distribuir las **4 noticias más relevantes de ciberseguridad** del mes.

### Fuentes
The Hacker News · SecurityWeek · Kaspersky Securelist · WeLiveSecurity ES

### Pasos
1. **Paso 1** – Cargar el código (GitHub, Google Drive o ZIP local)
2. **Paso 2** – Instalar dependencias
3. **Paso 3** – Persistencia con Google Drive _(opcional)_
4. **Paso 4** – Configurar credenciales vía Secrets de Colab
5. **Paso 5** – Ajustar parámetros de ejecución
6. **Paso 6** – Ejecutar el pipeline
7. **Paso 7** – Ver resultados en el notebook
8. **Paso 8** – Descargar archivos _(opcional)_

> 💡 **Tip**: Para SMTP, Teams, Slack u OpenAI ve al panel **🔑 Secrets** (barra lateral izquierda) y añade las claves antes de ejecutar el Paso 4.

In [ ]:
#@title 🔧 Paso 1 – Origen del proyecto { display-mode: "form" }
#@markdown Elige cómo cargar el código fuente y rellena el campo correspondiente.

import os
import sys
import shutil
import zipfile
from pathlib import Path

origen = 'GitHub'  #@param ["GitHub", "Google Drive (ZIP)", "Google Drive (carpeta)", "Subir ZIP local"]
github_url = 'https://github.com/dalesos92/cybernews_scraper.git'  #@param {type:"string"}
drive_zip_path = '/content/drive/MyDrive/scraper.zip'  #@param {type:"string"}
drive_folder_path = '/content/drive/MyDrive/scraper'  #@param {type:"string"}

# ── Helpers ─────────────────────────────────────────────────────────
EXTRACT_BASE = Path('/content/cybernews-scraper')

def _find_root(base):
    """Si el ZIP/carpeta contiene una sola subcarpeta, usarla como raíz."""
    items = [p for p in base.iterdir() if not p.name.startswith('.')]
    return items[0] if len(items) == 1 and items[0].is_dir() else base

# ── Cargar según el origen seleccionado ─────────────────────────────
if origen == 'GitHub':
    if EXTRACT_BASE.exists():
        shutil.rmtree(str(EXTRACT_BASE))
    ret = os.system(f'git clone {github_url} {EXTRACT_BASE}')
    if ret != 0:
        raise RuntimeError('Error al clonar. Verifica la URL y los permisos del repositorio.')
    PROJECT_DIR = EXTRACT_BASE

elif origen == 'Google Drive (ZIP)':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    if EXTRACT_BASE.exists():
        shutil.rmtree(str(EXTRACT_BASE))
    EXTRACT_BASE.mkdir(parents=True)
    with zipfile.ZipFile(drive_zip_path, 'r') as zf:
        zf.extractall(str(EXTRACT_BASE))
    PROJECT_DIR = _find_root(EXTRACT_BASE)

elif origen == 'Google Drive (carpeta)':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    if EXTRACT_BASE.exists():
        shutil.rmtree(str(EXTRACT_BASE))
    shutil.copytree(drive_folder_path, str(EXTRACT_BASE))
    PROJECT_DIR = _find_root(EXTRACT_BASE)

elif origen == 'Subir ZIP local':
    from google.colab import files as colab_files
    print('Selecciona el ZIP del proyecto (aparecerá un selector de archivos):')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No se subió ningún archivo.')
    zip_name = list(uploaded.keys())[0]
    if EXTRACT_BASE.exists():
        shutil.rmtree(str(EXTRACT_BASE))
    EXTRACT_BASE.mkdir(parents=True)
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall(str(EXTRACT_BASE))
    PROJECT_DIR = _find_root(EXTRACT_BASE)

# ── Configurar directorio de trabajo ────────────────────────────────
os.chdir(str(PROJECT_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f'✓ Directorio de trabajo: {os.getcwd()}')
print(f'  Archivos: {sorted(p.name for p in PROJECT_DIR.iterdir() if not p.name.startswith("."))[:12]}')


In [ ]:
#@title 📦 Paso 2 – Instalar dependencias

import subprocess
import os
from pathlib import Path

# Localizar requirements.txt: primero en el cwd establecido por el Paso 1,
# luego en la ruta fija de extracción, y finalmente en el directorio del notebook.
def _find_req():
    candidates = [
        Path(os.getcwd()) / 'requirements.txt',
        Path('/content/cybernews-scraper/requirements.txt'),
    ]
    try:
        candidates.insert(0, Path(PROJECT_DIR) / 'requirements.txt')
    except NameError:
        pass
    for p in candidates:
        if p.exists():
            return p
    return None

_req = _find_req()
if _req is None:
    raise FileNotFoundError(
        'No se encontró requirements.txt. '
        'Asegúrate de haber ejecutado el Paso 1 antes.'
    )

print(f'✓ requirements.txt encontrado en: {_req}')
print('Instalando dependencias...')
result = subprocess.run(
    ['pip', 'install', '-q', '-r', str(_req)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print('⚠ Advertencias / errores durante la instalación:')
    print((result.stderr or result.stdout)[-2000:])
else:
    print('✓ Dependencias instaladas correctamente')


In [ ]:
#@title 💾 Paso 3 – Persistencia en Google Drive (opcional)
#@markdown Activa esta opción para conservar la **base de datos SQLite** y los
#@markdown **archivos de salida** entre sesiones de Colab.
#@markdown Si no la activas, los datos se perderán al cerrar la sesión.

usar_drive = False  #@param {type:"boolean"}
drive_base = '/content/drive/MyDrive/CyberNewsScraper'  #@param {type:"string"}

from pathlib import Path

if usar_drive:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _db_dir = Path(drive_base) / 'data'
    _out_dir = Path(drive_base) / 'output'
    _db_dir.mkdir(parents=True, exist_ok=True)
    _out_dir.mkdir(parents=True, exist_ok=True)
    DB_PATH = str(_db_dir / 'cybernews.db')
    OUTPUT_DIR = str(_out_dir)
    print(f'✓ Base de datos → {DB_PATH}')
    print(f'✓ Directorio de salida → {OUTPUT_DIR}')
else:
    DB_PATH = 'data/cybernews.db'
    OUTPUT_DIR = 'output'
    print('✓ Usando almacenamiento temporal de Colab.')
    print('  (Los archivos se perderán al cerrar la sesión)')

In [ ]:
#@title 🔑 Paso 4 – Credenciales (Secrets de Colab)
#@markdown ### Cómo añadir secretos
#@markdown 1. Haz clic en el icono **🔑** de la barra lateral izquierda
#@markdown 2. Crea los secretos que necesites (son opcionales):
#@markdown
#@markdown | Nombre del secreto | Descripción |
#@markdown |---|---|
#@markdown | `SMTP_HOST` | Servidor SMTP (ej. `smtp.gmail.com`) |
#@markdown | `SMTP_USER` | Usuario o dirección de correo |
#@markdown | `SMTP_PASSWORD` | Contraseña o App Password de Google |
#@markdown | `SMTP_FROM` | Dirección remitente visible |
#@markdown | `EMAIL_RECIPIENTS` | Destinatarios separados por coma |
#@markdown | `TEAMS_WEBHOOK_URL` | URL del Incoming Webhook de Teams |
#@markdown | `SLACK_WEBHOOK_URL` | URL del Incoming Webhook de Slack |
#@markdown | `OPENAI_API_KEY` | API Key de OpenAI (resúmenes en español) |

import os

def _get_secret(name, default=''):
    """Lee desde Colab Secrets, con fallback a variables de entorno."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        return val if val is not None else default
    except Exception:
        return os.environ.get(name, default)

_SECRETS = {
    'SMTP_HOST':         _get_secret('SMTP_HOST'),
    'SMTP_PORT':         _get_secret('SMTP_PORT', '587'),
    'SMTP_USE_TLS':      'true',
    'SMTP_USER':         _get_secret('SMTP_USER'),
    'SMTP_PASSWORD':     _get_secret('SMTP_PASSWORD'),
    'SMTP_FROM':         _get_secret('SMTP_FROM'),
    'EMAIL_RECIPIENTS':  _get_secret('EMAIL_RECIPIENTS'),
    'TEAMS_WEBHOOK_URL': _get_secret('TEAMS_WEBHOOK_URL'),
    'SLACK_WEBHOOK_URL': _get_secret('SLACK_WEBHOOK_URL'),
    'OPENAI_API_KEY':    _get_secret('OPENAI_API_KEY'),
}

print('Estado de credenciales opcionales:')
_check_keys = [
    'SMTP_HOST', 'SMTP_USER', 'EMAIL_RECIPIENTS',
    'TEAMS_WEBHOOK_URL', 'SLACK_WEBHOOK_URL', 'OPENAI_API_KEY',
]
for k in _check_keys:
    icon = '\u2713' if _SECRETS.get(k) else '\u2014'
    label = 'configurado' if _SECRETS.get(k) else 'no configurado'
    print(f'  {icon} {k}: {label}')

In [ ]:
#@title ⚙️ Paso 5 – Parámetros de ejecución

modo = 'dry-run (solo genera archivos)'  #@param ["dry-run (solo genera archivos)", "normal (genera y guarda en BD)", "completo (genera, guarda y envía)"]

#@markdown ---
#@markdown #### Contenido
top_n = 4  #@param {type:"integer"}
lookback_days = 35  #@param {type:"integer"}

#@markdown #### Fuentes activas
enable_hackernews     = True   #@param {type:"boolean"}
enable_securityweek   = True   #@param {type:"boolean"}
enable_kaspersky      = True   #@param {type:"boolean"}
enable_welivesecurity = True   #@param {type:"boolean"}
enable_cybersecnews   = False  #@param {type:"boolean"}

print(f'✓ Modo: {modo}')
print(f'  top_n={top_n} | lookback_days={lookback_days}')
_fuentes = [
    ('HackerNews',   enable_hackernews),
    ('SecurityWeek', enable_securityweek),
    ('Kaspersky',    enable_kaspersky),
    ('WeLiveSec',    enable_welivesecurity),
    ('CyberSecNews', enable_cybersecnews),
]
print('  Fuentes: ' + ', '.join(f'{n}={"ON" if v else "off"}' for n, v in _fuentes))

In [ ]:
#@title 🚀 Paso 6 – Ejecutar pipeline

import os
from pathlib import Path

# ── Generar .env con todas las variables ────────────────────────────
_env_vars = {
    'LOG_LEVEL':             'INFO',
    'OUTPUT_DIR':            OUTPUT_DIR,
    'DB_PATH':               DB_PATH,
    'HTTP_TIMEOUT':          '30',
    'HTTP_MAX_RETRIES':      '3',
    'TOP_N':                 str(top_n),
    'LOOKBACK_DAYS':         str(lookback_days),
    'ENABLE_HACKERNEWS':     str(enable_hackernews).lower(),
    'ENABLE_SECURITYWEEK':   str(enable_securityweek).lower(),
    'ENABLE_KASPERSKY':      str(enable_kaspersky).lower(),
    'ENABLE_WELIVESECURITY': str(enable_welivesecurity).lower(),
    'ENABLE_CYBERSECNEWS':   str(enable_cybersecnews).lower(),
    **_SECRETS,
}
_env_content = '\n'.join(f'{k}={v}' for k, v in _env_vars.items())
Path('.env').write_text(_env_content, encoding='utf-8')
print('✓ .env generado')

# ── Construir argumentos CLI ─────────────────────────────────────────
_flags = []
if 'dry-run' in modo:
    _flags.append('--dry-run')
elif 'normal' in modo:
    _flags.append('--skip-send')
# 'completo' no necesita flags adicionales

_cmd = 'python main.py ' + ' '.join(_flags)
print(f'✓ Ejecutando: {_cmd}')
print('=' * 60)

# ── Ejecutar ─────────────────────────────────────────────────────────
os.system(_cmd)

In [ ]:
#@title 📊 Paso 7 – Ver resultados

import json
from pathlib import Path
from IPython.display import HTML, Markdown, display

_out = Path(OUTPUT_DIR)

# ── Resumen en texto ─────────────────────────────────────────────────
_json_path = _out / 'top4_monthly.json'
if _json_path.exists():
    _data = json.loads(_json_path.read_text(encoding='utf-8'))
    print(f"Generado : {_data['generated_at']}")
    print(f"Asunto   : {_data['subject']}")
    print()
    for item in _data['items']:
        print(f"  #{item['rank']} [{item['source']}]")
        print(f"     {item['title']}")
        print(f"     {item['url']}")
        print()
else:
    print('\u26a0 top4_monthly.json no encontrado. ¿Ejecutaste el Paso 6?')

# ── Markdown renderizado ─────────────────────────────────────────────
_md_path = _out / 'top4_monthly.md'
if _md_path.exists():
    print('\n' + '─' * 50)
    print('Vista Markdown')
    print('─' * 50)
    display(Markdown(_md_path.read_text(encoding='utf-8')))

# ── Preview HTML del email ───────────────────────────────────────────
_html_path = _out / 'top4_email.html'
if _html_path.exists():
    print('\n' + '─' * 50)
    print('Preview HTML del email')
    print('─' * 50)
    display(HTML(_html_path.read_text(encoding='utf-8')))

In [ ]:
#@title 💾 Paso 8 – Descargar archivos de salida (opcional)
#@markdown Descarga los artefactos generados a tu máquina local.

from pathlib import Path

try:
    from google.colab import files as colab_files
    _out = Path(OUTPUT_DIR)
    _found = 0
    for _fname in ['top4_monthly.json', 'top4_monthly.md', 'top4_email.html']:
        _fpath = _out / _fname
        if _fpath.exists():
            colab_files.download(str(_fpath))
            print(f'\u2713 Descargando {_fname}')
            _found += 1
    if _found == 0:
        print('\u26a0 No se encontraron archivos. Ejecuta el Paso 6 primero.')
except ImportError:
    print('\u2139 Esta celda solo funciona en Google Colab.')